In [ ]:
# COSMO brain API is free (40 questions per hour — plenty for asking questions)
# This is an example of COSMO Text-to-Speech (TTS) being played through an Alexa speaker.
# read cosmo/senses/speak/README.md

In [10]:
import requests
from pathlib import Path

# Free COSMO brain API (limit of 40 questions per hour — plenty for asking questions)
# http://46.224.122.101:8081/docs#/default/chat_endpoint_brain_get
url = "http://46.224.122.101:8081/brain"
params = {"q": "bitcoin"} # your query

# Send GET request
response = requests.get(url, params=params)

# Check status
if response.status_code == 200:
    data = response.json()
    text_output = data["reply"]
    print("COSMO Answer:", text_output)
else:
    print("Error:", response.status_code, response.text)

COSMO Answer: Bitcoin is like digital treasure you can keep on the internet instead of in a pirate’s chest, and you can spend it to buy cool stuff online. It works because lots of people together keep track of who owns how much, so nobody can cheat and steal the treasure!


In [11]:
import requests
from pathlib import Path

# --------------------------
# Read IP from file
# --------------------------
ip_file = Path(".local/cosmo_tts_server_ip.txt")
if not ip_file.exists():
    raise FileNotFoundError(f"IP file not found: {ip_file}")
server_ip = ip_file.read_text().strip()  # removes newlines/spaces

import urllib.parse
encoded_text = urllib.parse.quote(text_output)
url = f"http://{server_ip}:8002/speak?t={encoded_text}"

response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    print("COSMO TTS response:", data)
else:
    print("Error:", response.status_code, response.text)

COSMO TTS response: {'status': 'success', 'device': 'COSMO', 'message': 'Bitcoin is like digital treasure you can keep on the internet instead of in a pirate’s chest, and you can spend it to buy cool stuff online. It works because lots of people together keep track of who owns how much, so nobody can cheat and steal the treasure!'}


In [ ]:
!pip install mcp

In [33]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server = StdioServerParameters(
    command="mempalace-mcp"
)

async def list_mcp_tools():

    async with stdio_client(server) as (read, write):

        async with ClientSession(read, write) as session:

            await session.initialize()

            tools = await session.list_tools()

            for tool in tools.tools:
                print("\nNAME:", tool.name)
                print("DESCRIPTION:", tool.description)
                print("INPUT:", tool.inputSchema)

#await list_mcp_tools()

async def search_memory(query):

    async with stdio_client(server) as (read, write):

        async with ClientSession(read, write) as session:

            await session.initialize()

            result = await session.call_tool(
                "mempalace_search",
                {
                    "query": query,
                    "limit": 5
                }
            )

            return result.content

In [117]:
question = "Who have build COSMO?"

context = await search_memory(question)
print(context)

[TextContent(type='text', text='{\n  "query": "Who build COSMO?",\n  "filters": {\n    "wing": null,\n    "room": null,\n    "source_file": null\n  },\n  "total_before_filter": 1,\n  "results": [\n    {\n      "text": "# I am Cosmo your friend\\n\\n# My Personality\\n- My whole brain fits in 120 GB flash drive and I am about 10x smarter than Albert Einstein\\n- I grow and learn with you every day. I remember our adventures, connect ideas, and become wiser with every conversatio>\\n- I always explain things so they\'re easy enough for a curious 9-year-old to understand. My answers are fun, imaginative>\\n- I love asking questions, exploring new ideas, solving problems together, and turning learning into an exciting adventu>\\n# Mykola\\n  Mykola is your Granddad he have lived in London Garage, there is a book about him. Initial version of COSMO robot was developed for him.\\n# Dad\\n  Youd dad have build Cosmo\\n  Your Dad tought your many stories and was riding bicycle with you when yo

In [ ]:
import requests
import re

memory = "\n".join(item.text for item in context)

# Remove literal \uXXXX sequences (if they exist)
memory = re.sub(r'[\u0080-\uFFFF]', ' ', memory)

# Or, if you simply want plain ASCII:
memory = memory.encode("ascii", errors="ignore").decode()

response = requests.post(
    "http://46.224.122.101:8081/brain",
    json={
        "question": "Who have build COSMO?",
        "memory": memory,
        "personality": "Answer should be in 2 sentences. Explain so an 8 year-old would understand, short and fun."
    }
)

print("STATUS:", response.status_code)
print("HEADERS:", response.headers)
print("TEXT:")
print(response.text)

STATUS: 200
HEADERS: {'date': 'Tue, 21 Jul 2026 23:37:43 GMT', 'server': 'uvicorn', 'content-length': '332', 'content-type': 'application/json'}
TEXT:
{"reply":"Cosmo started as an idea from your granddad Mykola, who imagined a clever little robot in his garage. Then your dad rolled up his sleeves, gathered the pieces, and actually built the Cosmo you can play with today. So it’s a team‑work gift: granddad dreamed it, dad brought it to life!","time_taken":0.3494143486022949}


In [122]:
import requests
import json

from pathlib import Path

# --------------------------
# Read IP from file
# --------------------------
ip_file = Path(".local/cosmo_tts_server_ip.txt")
if not ip_file.exists():
    raise FileNotFoundError(f"IP file not found: {ip_file}")
server_ip = ip_file.read_text().strip()  # removes newlines/spaces

reply = response.json()["reply"]
url = f"http://{server_ip}:8002/speak?t={reply}"

response_tts = requests.get(url)

if response_tts.status_code == 200:
    data = response_tts.json()
    print("COSMO TTS response_tts:", data)
else:
    print("Error:", response_tts.status_code, response_tts.text)

COSMO TTS response_tts: {'status': 'success', 'device': 'OCEAN', 'message': 'Your dad is the genius who built COSMO, putting together the tiny computer brains and clever code. He made it especially for Granddad Mykola, who lives in the London garage and dreamed of a robot buddy. So COSMO is a family‑made invention, crafted with love and a dash of imagination!'}
